<a href="https://colab.research.google.com/github/esther148/Ai-lab-1/blob/main/Fake%20news%20detection%20prediction%20using%20logistic%20regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import nltk

In [ ]:
df = pd.read_csv('/content/sample_data/fake_news_dataset.csv')
print(df.head())

                                  title  \
0               Foreign Democrat final.   
1   To offer down resource great point.   
2          Himself church myself carry.   
3                  You unit its should.   
4  Billion believe employee summer how.   

                                                text        date    source  \
0  more tax development both store agreement lawy...  2023-03-10  NY Times   
1  probably guess western behind likely next inve...  2022-05-25  Fox News   
2  them identify forward present success risk sev...  2022-09-01       CNN   
3  phone which item yard Republican safe where po...  2023-02-07   Reuters   
4  wonder myself fact difficult course forget exa...  2023-04-03       CNN   

                 author    category label  
0          Paula George    Politics  real  
1           Joseph Hill    Politics  fake  
2        Julia Robinson    Business  fake  
3  Mr. David Foster DDS     Science  fake  
4         Austin Walker  Technology  fake  


In [ ]:
print(df.duplicated().sum())
print(df.isnull().sum())
# The original df was inadvertently reassigned here. Keeping df as DataFrame.
# If you intend to drop nulls, it should be df.dropna(inplace=True)
# For now, removing the lines that change df's structure.

0
title          0
text           0
date           0
source      1000
author      1000
category       0
label          0
dtype: int64


In [ ]:
df['context'] = df['title'].astype(str)+ ""+df['text'].astype(str)

In [ ]:
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
nltk.download('punkt')
stemmer = PorterStemmer()
def stem_text(context):
  if not isinstance(context, str):
    context = str(context)
    tokens = word_tokenize(context.lower())
    stems = [stemmer.stem(token) for token in tokens]
    return"".join(stems)
    df['combined_stemmed'] = df['context'].apply(stem_text)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['label'] = le.fit_transform(df['label'])
print (df)

                                       title  \
0                    Foreign Democrat final.   
1        To offer down resource great point.   
2               Himself church myself carry.   
3                       You unit its should.   
4       Billion believe employee summer how.   
...                                      ...   
19995                      House party born.   
19996  Though nation people maybe price box.   
19997        Yet exist with experience unit.   
19998               School wide itself item.   
19999         Offer chair cover senior born.   

                                                    text        date  \
0      more tax development both store agreement lawy...  2023-03-10   
1      probably guess western behind likely next inve...  2022-05-25   
2      them identify forward present success risk sev...  2022-09-01   
3      phone which item yard Republican safe where po...  2023-02-07   
4      wonder myself fact difficult course forget exa...  2023-

In [ ]:
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
df_source_encoded = ohe.fit_transform(df[['source']])
df_category_encoded = ohe.fit_transform(df[['category']])

# The encoded features are now in df_source_encoded and df_category_encoded.
# If you want to add them back to the original DataFrame, you would typically do:
# df = pd.concat([df.drop(columns=['source', 'category']),
#                   pd.DataFrame(df_source_encoded, columns=ohe.get_feature_names_out(['source'])),
#                   pd.DataFrame(df_category_encoded, columns=ohe.get_feature_names_out(['category']))], axis=1)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(stop_words='english',ngram_range=(1,2), max_features=5000)
x_text=vectorizer.fit_transform(df['context'])

In [ ]:
from scipy.sparse import hstack
x = hstack([x_text,df_source_encoded,df_category_encoded])
y= df['label']

In [ ]:
x_train,x_test,y_train,y_test = train_test_split(x, y, test_size = 0.2 , random_state = 42)

In [ ]:
model = RandomForestClassifier(n_estimators=200, random_state=42)
model = model.fit(x_train,y_train)

In [ ]:
y_pred =model.predict(x_test)

In [ ]:
from sklearn.metrics import accuracy_score
acc= accuracy_score(y_test,y_pred)
print(acc)

0.5025
